**Questões**
1. Qual configuração do menu lateral levou mais usuários a acessarem a página específica?
2. Qual o tempo médio de sessão para cada configuração do menu?
3. Qual é a taxa de conversão para cada configuração do menu?
4. A diferença nos acessos entre as configurações é estatisticamente significativa?
5. Quais insights podem ser derivados dos dados para otimizar a navegação do site e aumentar o engajamento dos usuários?
6. Existe alguma tendência temporal (comportamento ao longo dia) nos acessos ou conversões para cada configuração do menu?
7. Quais recomendações você faria com base nos dados analisados para otimizar a navegação do site?

## 1. Configurações e Bibliotecas

Abaixo realiza-se a configuração inicial do ambiente e a importação das bibliotecas utilizadas nesta análise:
* **Pandas:** Para manipulação e agregação dos dados.
* **IpyDataGrid:** Para visualização interativa das tabelas de dados.
* **SciPy:** Para a aplicação do teste de significância estatística.


*Nota: O comando de instalação (`%pip`) está comentado por padrão para agilizar as execuções subsequentes. Descomente a linha caso seja a primeira vez executando este notebook.*

In [ ]:
# A linha linha de comando a seguir instala as bibliotecas listadas em requirements.txt.
# %pip install -r requirements.txt -q
# Obs.: Remover o comentário da linha acima no primeiro uso ou para atualizar as bibliotecas.

import pandas as pd
from scipy.stats import chi2_contingency
from itables import show
from ipydatagrid import DataGrid, TextRenderer, Expr



## 2. Carregamento e tratamento dos dados

A seguir os dados do teste A/B dispostos na planilha _ab_test_data.xlsx_ serão importados como o dataset `dados`.


In [92]:
# Importando os dados do arquivo Excel e exibindo-os em formato de tabela interativa
dados = pd.read_excel("ab_test_data.xlsx")
show(dados, paging=True, lengthMenu=[15, 30, 50], allow_html=True)


# Contagem de valores únicos da coluna `user_id`
total_usuarios_unicos = dados['user_id'].nunique()
print(f"\nTotal de usuários únicos: {total_usuarios_unicos} ---------------")


# Exibindo as informações estruturais e estatística descritiva
print(f"\nInformações estruturais ----------------------\n")
dados.info()
print(f"-"*45)

Loading ITables v2.8.0 from the internet... (need help?)



Total de usuários únicos: 1295 ---------------

Informações estruturais ----------------------

<class 'pandas.DataFrame'>
RangeIndex: 2305 entries, 0 to 2304
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           2305 non-null   int64         
 1   device            2305 non-null   str           
 2   country           2305 non-null   str           
 3   timestamp         2305 non-null   datetime64[us]
 4   group             2305 non-null   str           
 5   converted         2305 non-null   int64         
 6   session_duration  2269 non-null   float64       
 7   hour_of_day       2305 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 144.2 KB
---------------------------------------------



A partir da inspeção acima, foram identificados os seguintes pontos que necessitam atenção:

- **Tipagem:** A coluna de `user_id` será convertida para _string_, já `converted` será convertida em booleana (True/False);

- **Redundância:** A coluna `hour_of_day` é redundante com `timestamp` e será removida;

- **Usuários duplicados:** Múltiplos registros para do mesmo `user_id` em diferentes países, indicando usuários retornantes ou possíveis falhas na coleta;

- **Valores nulos:** Sessões com duração nula (`NaN`).




A seguinte etapa busca observar o histórico de acessos irregulares agrupando por `user_id`, ordenando a base pelo `timestamp` e destacando valores nulos em `session_duration`.

In [ ]:
# i. Convertendo tipos de dados e removendo colunas
dados["converted"] = dados["converted"].astype(bool)
dados["user_id"] = dados["user_id"].astype(str)
dados = dados.drop(columns=["hour_of_day"])


# ii. Identificando ids de usuários duplicados
ids_duplicados = dados[dados.duplicated('user_id', keep=False)]['user_id'].unique()


# iii. Identificando usuários com mais de um país
ids_multipais = dados.groupby('user_id')['country'].nunique()
ids_multipais = ids_multipais[ids_multipais > 1].index.tolist()


# iv. Identificando usuários com ao menos um NaN em 'session_duration'
ids_com_nan = dados[dados['session_duration'].isna()]['user_id'].unique()


# v. Interseção: Usuários que são DUPLICADOS E MULTIPAÍS E COM NAN
ids_alvo = list(set(ids_duplicados) & set(ids_multipais) & set(ids_com_nan))


# vi. Filtrando a base original e ordenar
dados_alvo = dados[dados['user_id'].isin(ids_alvo)].copy()
dados_alvo['id_temp'] = dados_alvo['user_id'].astype(int)
dados_sorted_alvo = dados_alvo.sort_values(by=['id_temp', 'timestamp']).drop(columns=['id_temp'])


# vii. Exibindo tabela com observações contendo user_id duplicados, multipaís e NaNs
def destacar_linhas_nan(df):
    return df.style.apply(
        lambda row: ['color: red'] * len(row) 
        if pd.isna(row['session_duration']) else [''] * len(row), 
        axis=1
    )
show(destacar_linhas_nan(dados_sorted_alvo), paging=True, lengthMenu=[15, 30, 50], allow_html=True)


# viii. Contagem de usuários únicos na base filtrada
total_usuarios_alvo = dados_sorted_alvo['user_id'].nunique()
print(f"\nTotal de usuários únicos na base filtrada: {total_usuarios_alvo}")
lista_usuarios_alvo = dados_sorted_alvo['user_id'].unique().tolist()
print(lista_usuarios_alvo)
print(f"-"*45)

Loading ITables v2.8.0 from the internet... (need help?)



Total de usuários únicos na base filtrada: 16
['37', '47', '575', '700', '861', '946', '957', '967', '1123', '1137', '1151', '1161', '1301', '1397', '1824', '1979']




A partir da identificação de irregularidades no dataset (duplicidade + multiplicidade de países + falhas de coleta) foi possível concluir que houve falhas sistêmicas para 16 usuários específicos. 

Para garantir a integridade dos dados do teste A/B são sugeridas as seguintes alterações:

- **Padronização geográfica (_first-touch_):** Corrigir registros com conflito de localização na coluna `country` priorizando o primeiro acesso;

- **Limpeza de integridade:** Exclusão dos registros que contiverem valores nulos (NaN) na coluna `session_duration`.



Abaixo, realiza-se a aplicação destas alterações e a exportação do dataset consolidado para criação da dashboard no DataStudio (antigo Looker) .

In [ ]:
# i. Padronização geográfica (first-touch)
mapa_pais_correto = (
    dados.dropna(subset=['country'])
    .sort_values('timestamp')
    .groupby('user_id')['country']
    .first()
)
dados_tratados = dados.copy()
dados_tratados['country'] = dados_tratados['user_id'].map(mapa_pais_correto)

# ii. Limpeza de integridade (Remoção de NaNs em session_duration)
dados_finais = dados_tratados.dropna(subset=['session_duration']).copy()

# iii. Exportação para CSV (pronto para o Looker Studio)
dados_finais.to_csv("ab_test_data_final.csv", index=False)

print(f"Base Final: {len(dados_finais)} registros;")
print("Dataset exportado com sucesso para 'ab_test_data_final.csv';")
print(f"Tratamento concluído!")



Base Final: 2269 registros;
Dataset exportado com sucesso para 'ab_test_data_final.csv';
Tratamento concluído!
